In [1]:
%pip install python-dotenv requests



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
from dotenv import load_dotenv
load_dotenv()
KEY: str = os.environ["KOREAN_DICT_KEY"]
print(KEY[:4] + "***")

0BEF***


In [25]:
# Q1 (a)
import time
import json
import requests

def search_word(q: str, num: int = 10, start: int = 1) -> dict:
    params = {
        "key": KEY, "q": q, "req_type": "json",
        "num": num, "start": start, "type1": "word"
    }
    r = requests.get(
        "https://opendict.korean.go.kr/api/search",
        params=params, timeout=10
    )
    r.raise_for_status()
    return r.json()

# Q1 (a)
requests.get()을 사용하여 주어진 매개변수와 함께 API를 호출하고 타임아웃을 10초로 지정했습니다. r.raise_for_status()로 통신 오류를 검사한 후, r.json()을 통해 응답을 파이썬 딕셔너리로 파싱하여 반환합니다.

In [26]:
# Q1 (b)
data = search_word("김치")
print(json.dumps(data, ensure_ascii=False, indent=2)[:400])

{
  "channel": {
    "total": 328,
    "num": 10,
    "title": "우리말샘 개발 지원(Open API) - 사전 어휘 검색",
    "start": 1,
    "description": "우리말샘 개발 지원(Open API) - 사전 어휘 검색 결과",
    "link": "https://opendict.korean.go.kr",
    "item": [
      {
        "word": "김치",
        "sense": [
          {
            "syntacticArgument": "",
            "syntacticAnnotation": "",
            "cat": "",
          


# Q1 (b)
코드 실행 결과
{
  "channel": {
    "total": 328,
    "num": 10,
    "title": "우리말샘 개발 지원(Open API) - 사전 어휘 검색",
    "start": 1,
    "description": "우리말샘 개발 지원(Open API) - 사전 어휘 검색 결과",
    "link": "https://opendict.korean.go.kr",
    "item": [
      {
        "word": "김치",
        "sense": [
          {
            "syntacticArgument": "",
            "syntacticAnnotation": "",
            "cat": "",
          
설명
ensure_ascii=False를 빼면 기본값(True)이 적용되어 한글이 \uc5b8\uc5b4와 같은 이스케이프 형태로 변환되므로 원문 그대로 출력되지 않습니다.

In [27]:
# Q1 (c)
total = data["channel"]["total"]
items = data["channel"].get("item", [])
n = len(items)

print(f"총 {total}건, 이 페이지 {n}건")

for it in items[:5]:
    word = it["word"]
    pos = it.get("pos", "품사 없음")
    
    sense = it["sense"]
   
    if isinstance(sense, list):
        sense = sense[0] 
        
    definition = sense["definition"]
    print(f"{word} ({pos}) --> {definition[:40]}")

# Gemini 사용 링크:"https://gemini.google.com/share/0c732dca0316"

총 328건, 이 페이지 10건
김치 (품사 없음) --> 소금에 절인 배추나 무 따위를 고춧가루, 파, 마늘 따위의 양념에 버무린
김-치 (품사 없음) --> 고려 말기·조선 초기의 문신(?~?). 자는 기보(基甫). 김해 부사를 
김-치 (품사 없음) --> 조선 중기의 문신(1577~1625). 자는 사정(士精). 호는 남봉(南
김치 공장 (품사 없음) --> 김치를 만드는 공장.
김치 보릿고개 (품사 없음) --> 김장철인 가을·겨울과 달리 상대적으로 김치가 부족한 봄여름을 비유적으로 


# Q1 (c)
코드 실행 결과
총 328건, 이 페이지 10건
김치 (품사 없음) --> 소금에 절인 배추나 무 따위를 고춧가루, 파, 마늘 따위의 양념에 버무린
김-치 (품사 없음) --> 고려 말기·조선 초기의 문신(?~?). 자는 기보(基甫). 김해 부사를 
김-치 (품사 없음) --> 조선 중기의 문신(1577~1625). 자는 사정(士精). 호는 남봉(南
김치 공장 (품사 없음) --> 김치를 만드는 공장.
김치 보릿고개 (품사 없음) --> 김장철인 가을·겨울과 달리 상대적으로 김치가 부족한 봄여름을 비유적으로 

설명
전체 결과 수와 항목을 먼저 추출하고, pos 값이 없는 경우 에러가 나지 않도록 dict.get을 써서 "품사 없음"으로 처리했습니다. 또한 sense 데이터가 딕셔너리가 아닌 리스트로 들어오는 경우도 있어서, isinstance로 형태를 확인하고 첫 번째 뜻풀이만 가져오도록 코드를 작성했습니다.
기존에 작성한 코드   sense = it["sense"]["definition"]에서 Typeerror가 발생해, gemini를 이용해 해결했습니다. (Gemini 사용 링크:"https://gemini.google.com/share/0c732dca0316")

In [28]:
# Q2 
from collections import Counter

words: list[str] = [
    "김치", "라면", "만두", "김밥", 
    "국수", "떡볶이", "불고기", "비빔밥"
]
all_items = []

# (a) 
for w in words:
    result = search_word(w)
    total = result["channel"]["total"]
    print(f"{w}: {total}건")
    
    items = result["channel"].get("item", [])
    all_items.extend(items)
    
    time.sleep(0.3)

# (b) 
pos_list = [it.get("pos") or "(미상)" for it in all_items]
cnt = Counter(pos_list)

print(cnt.most_common(3))

김치: 328건
라면: 86건
만두: 89건
김밥: 39건
국수: 227건
떡볶이: 24건
불고기: 38건
비빔밥: 38건
[('(미상)', 80)]


# Q2 
코드 실행 결과
김치: 328건
라면: 86건
만두: 89건
김밥: 39건
국수: 227건
떡볶이: 24건
불고기: 38건
비빔밥: 38건
[('(미상)', 80)]

설명
(a) 8개의 음식 검색어를 순회하며 search_word() 함수를 호출해 각 검색어의 전체 결과 수를 출력했습니다. 서버에 과부하를 주거나 차단당하지 않도록 매 요청 사이에 time.sleep(0.3)을 추가하여 스크래핑 매너를 지켰습니다.
(b)모든 항목의 품사 데이터를 리스트로 모은 뒤, collections.Counter의 most_common(3)을 사용하여 빈도 상위 3개를 추출했습니다. 품사 정보가 없는 항목은 get()을 활용해 "(미상)"으로 처리했습니다.
검색어로 사용된 단어들이 모두 음식 이름이므로,가장 흔한 품사는 '명사'입니다.